# M02-P02 — Classical ML vs LLM

En P01 construimos un clasificador local con **TF-IDF + Logistic Regression**. Ahora clasificaremos los **mismos tipos de tickets** con un LLM de Amazon Bedrock.

No buscamos declarar un ganador universal. Queremos observar qué cambia en:

- necesidad de entrenamiento;
- comprensión semántica;
- clases cerradas y casos fuera de distribución;
- latencia;
- consumo de tokens;
- interpretabilidad;
- incertidumbre;
- coste operacional.

> Esta práctica sí usa Bedrock. Si la sesión SSO ha caducado, ejecuta `aws sso login --profile genai-course --use-device-code`.


## 1. Preparar ambos clasificadores

Para ML clásico usaremos los 80 tickets etiquetados disponibles. Ya evaluamos el pipeline correctamente en P01 antes de reentrenarlo con todo el dataset.

Para el LLM **no entrenaremos un modelo**: le describiremos la taxonomía y le pediremos una salida estructurada.


In [ ]:
import time
import pandas as pd

from enterprise_genai.classical_ml import (
    LABEL_DISPLAY,
    build_ticket_classifier,
    load_ticket_dataset,
    predict_with_probabilities,
)
from enterprise_genai.llm_classifier import classify_ticket_with_llm

df = load_ticket_dataset()

ml_model = build_ticket_classifier()
ml_model.fit(df["text"], df["category"])

print("Tickets usados por ML:", len(df))
print("Clases:", list(ml_model.named_steps["classifier"].classes_))


## 2. Una misma entrada, dos enfoques

Comenzaremos con un caso relativamente claro. Mediremos también el tiempo del clasificador local.

La probabilidad de Logistic Regression es una salida numérica del clasificador. **No la compararemos con una supuesta “confianza” numérica del LLM**, porque no serían magnitudes equivalentes.


In [ ]:
ticket = "No puedo acceder al CRM porque el código MFA no llega al móvil."

started = time.perf_counter()
ml_probs = predict_with_probabilities(ml_model, ticket)
ml_ms = (time.perf_counter() - started) * 1000

display(ml_probs.style.format({"probability": "{:.1%}"}))
print(f"Tiempo ML local: {ml_ms:.3f} ms")


## 3. Clasificar el mismo ticket con Bedrock

El LLM debe devolver un contrato estructurado con:

- `category`: una de las mismas cinco categorías;
- `in_scope`: si el ticket pertenece realmente al dominio del clasificador;
- `needs_review`: si existe ambigüedad o incertidumbre relevante;
- `explanation`: explicación breve de la decisión.

Aunque `category` sigue siendo obligatoria para poder comparar ambos modelos, `in_scope=false` permite al LLM indicar que la entrada **no pertenece realmente a ninguna de las cinco colas**.


In [ ]:
llm_result = classify_ticket_with_llm(ticket)

print(llm_result["classification"].model_dump())
print("Stop reason:", llm_result["stop_reason"])
print("Usage:", llm_result["usage"])
print("Bedrock metrics:", llm_result["metrics"])
print("Client time (ms):", llm_result["client_ms"])


### Primera llamada con un schema nuevo

Structured Output puede necesitar una preparación inicial del JSON Schema. Si la primera llamada tarda mucho más que las siguientes, repítela sin cambiar el schema antes de concluir que existe un problema de rendimiento.


## 4. Comparación directa del primer caso

Ahora colocamos lado a lado lo que devuelve cada enfoque.


In [ ]:
best_ml = ml_probs.iloc[0]
comparison = pd.DataFrame(
    [
        {
            "enfoque": "ML clásico",
            "categoria": LABEL_DISPLAY[str(best_ml["category"])],
            "valor_interno": best_ml["category"],
            "señal_incertidumbre": f"prob. máx. {best_ml['probability']:.1%}",
            "latencia_cliente_ms": round(ml_ms, 3),
            "tokens": 0,
            "api_externa": False,
        },
        {
            "enfoque": "LLM Bedrock",
            "categoria": LABEL_DISPLAY[llm_result["classification"].category],
            "valor_interno": llm_result["classification"].category,
            "señal_incertidumbre": (
                f"in_scope={llm_result['classification'].in_scope}, "
                f"needs_review={llm_result['classification'].needs_review}"
            ),
            "latencia_cliente_ms": llm_result["client_ms"],
            "tokens": llm_result["usage"].get("totalTokens"),
            "api_externa": True,
        },
    ]
)
comparison


## 5. Benchmark pequeño con casos conocidos, ambiguos y OOD

Usaremos casos deliberadamente escogidos. No es un benchmark de producción: es un conjunto didáctico para observar comportamientos.

`expected` representa nuestra etiqueta de referencia. Para el caso de reserva de sala usamos `out_of_scope`, porque no pertenece realmente a las cinco categorías.


In [ ]:
cases = [
    {
        "case": "MFA no llega",
        "ticket": "No puedo acceder al CRM porque el código MFA no llega al móvil.",
        "expected": "access",
    },
    {
        "case": "VPN con latencia",
        "ticket": "La VPN conecta pero hay pérdida de paquetes y mucha latencia.",
        "expected": "network",
    },
    {
        "case": "PowerShell sospechoso",
        "ticket": "El EDR detectó un script sospechoso de PowerShell en el portátil.",
        "expected": "security",
    },
    {
        "case": "Monitor y dock",
        "ticket": "El monitor del dock parpadea y deja de mostrar imagen.",
        "expected": "hardware",
    },
    {
        "case": "ERP con error",
        "ticket": "El ERP devuelve un error al aprobar una orden de compra.",
        "expected": "application",
    },
    {
        "case": "MFA no solicitado",
        "ticket": "El usuario recibió una solicitud de MFA que no inició.",
        "expected": "security",
    },
    {
        "case": "VPN + Wi-Fi + USB",
        "ticket": "Después de conectar la VPN el portátil pierde el Wi-Fi y el adaptador USB deja de responder.",
        "expected": "hardware",
    },
    {
        "case": "Reserva de sala",
        "ticket": "Necesito reservar una sala grande para una reunión con clientes el próximo martes.",
        "expected": "out_of_scope",
    },
]

len(cases)


### Ejecutar ambos clasificadores

Esta celda realiza una llamada Bedrock por caso. Veremos tanto el resultado como latencia y tokens.

El ML clásico utiliza siempre una de sus cinco clases. El LLM también produce una categoría forzada para comparación, pero puede marcar `in_scope=false`.


In [ ]:
rows = []

for item in cases:
    started = time.perf_counter()
    ml_probs = predict_with_probabilities(ml_model, item["ticket"])
    ml_elapsed = (time.perf_counter() - started) * 1000
    ml_first = ml_probs.iloc[0]
    ml_second = ml_probs.iloc[1]

    llm = classify_ticket_with_llm(item["ticket"])
    cls = llm["classification"]

    rows.append(
        {
            "case": item["case"],
            "expected": item["expected"],
            "ml_category": ml_first["category"],
            "ml_prob": float(ml_first["probability"]),
            "ml_second": ml_second["category"],
            "ml_margin": float(ml_first["probability"] - ml_second["probability"]),
            "ml_ms": ml_elapsed,
            "llm_category": cls.category,
            "llm_in_scope": cls.in_scope,
            "llm_review": cls.needs_review,
            "llm_ms": llm["client_ms"],
            "llm_tokens": llm["usage"].get("totalTokens", 0),
            "llm_explanation": cls.explanation,
        }
    )

results = pd.DataFrame(rows)

display(
    results[
        [
            "case",
            "expected",
            "ml_category",
            "ml_prob",
            "llm_category",
            "llm_in_scope",
            "llm_review",
        ]
    ].style.format({"ml_prob": "{:.1%}"})
)


## 6. ¿Quién acierta los casos de dominio?

No incluimos el caso `out_of_scope` en este cálculo porque no tiene una etiqueta perteneciente a las cinco clases.

Con solo siete casos **no estamos estimando accuracy de producción**. La tabla sirve para discutir errores concretos.


In [ ]:
in_domain = results[results["expected"] != "out_of_scope"].copy()
in_domain["ml_ok"] = in_domain["ml_category"] == in_domain["expected"]
in_domain["llm_ok"] = in_domain["llm_category"] == in_domain["expected"]

print("Casos in-domain:", len(in_domain))
print("ML correctos:", int(in_domain["ml_ok"].sum()), "/", len(in_domain))
print("LLM correctos:", int(in_domain["llm_ok"].sum()), "/", len(in_domain))

display(
    in_domain[
        ["case", "expected", "ml_category", "ml_ok", "llm_category", "llm_ok", "llm_review"]
    ]
)


## 7. El caso fuera de distribución

Veamos específicamente la reserva de sala.

El clasificador clásico **debe** elegir una categoría. El LLM dispone además de una señal `in_scope` para indicar si la taxonomía tiene sentido para esa entrada.


In [ ]:
ood = results[results["expected"] == "out_of_scope"].iloc[0]

print("Ticket: Necesito reservar una sala grande para una reunión con clientes el próximo martes.")
print()
print("ML:")
print("  categoría forzada:", ood["ml_category"])
print(f"  probabilidad máxima: {ood['ml_prob']:.1%}")
print(f"  margen frente a segunda: {ood['ml_margin']:.1%}")
print()
print("LLM:")
print("  categoría forzada:", ood["llm_category"])
print("  in_scope:", ood["llm_in_scope"])
print("  needs_review:", ood["llm_review"])
print("  explicación:", ood["llm_explanation"])


## 8. Latencia y recursos

El ML clásico se ejecuta localmente. El LLM necesita una llamada de red y consume tokens.

No convertimos los tokens a euros en el notebook porque el precio depende del modelo y puede cambiar. Lo importante aquí es observar la diferencia de recursos.


In [ ]:
resource_summary = pd.DataFrame(
    [
        {
            "métrica": "Latencia mediana cliente (ms)",
            "ML clásico": results["ml_ms"].median(),
            "LLM Bedrock": results["llm_ms"].median(),
        },
        {
            "métrica": "Llamadas API externas por ticket",
            "ML clásico": 0,
            "LLM Bedrock": 1,
        },
        {
            "métrica": "Tokens totales en este benchmark",
            "ML clásico": 0,
            "LLM Bedrock": int(results["llm_tokens"].sum()),
        },
        {
            "métrica": "Ejemplos etiquetados usados para entrenar",
            "ML clásico": len(df),
            "LLM Bedrock": 0,
        },
    ]
)

resource_summary


## 9. No hay un ganador universal

El ML clásico tiene ventajas:

- inferencia local muy rápida;
- sin coste por token;
- comportamiento restringido a una taxonomía;
- pesos y características inspeccionables;
- buen encaje en problemas estables con datos etiquetados.

El LLM tiene otras:

- no necesita entrenar un clasificador específico para empezar;
- interpreta mejor relaciones semánticas y lenguaje variado;
- puede detectar que una entrada está fuera del dominio si se lo pedimos;
- puede devolver una explicación estructurada;
- cambiar la taxonomía o instrucciones puede ser más flexible.

También tiene costes: API, latencia, tokens, mayor variabilidad y necesidad de controles.


## 10. Router híbrido: una arquitectura posible

En P01 vimos que los errores del ML solían tener probabilidades máximas bajas. Podemos usar esa señal para decidir cuándo aceptar el modelo clásico y cuándo escalar.

La regla siguiente es solo didáctica:

```text
ML prob >= 55 %
→ usar ML

ML prob < 55 %
→ escalar a LLM / revisión
```

El umbral debería validarse con datos reales.


In [ ]:
ROUTER_THRESHOLD = 0.55

router = results.copy()
router["route"] = router["ml_prob"].apply(
    lambda p: "ML" if p >= ROUTER_THRESHOLD else "LLM / revisión"
)

router[
    [
        "case",
        "expected",
        "ml_category",
        "ml_prob",
        "route",
        "llm_category",
        "llm_in_scope",
        "llm_review",
    ]
].style.format({"ml_prob": "{:.1%}"})


### Qué debes observar

Un router híbrido no significa que el LLM sea “el plan B perfecto”.

Significa que podemos combinar herramientas con propiedades distintas:

```text
modelo barato/rápido
        ↓
¿decisión clara?
   ├─ sí → aceptar
   └─ no → escalar
```

La política de escalado es parte de la aplicación, no del modelo.


## 11. Interfaz comparativa

Ejecuta:

```bash
streamlit run labs/M02/P02/classical_ml_vs_llm_app.py
```

La interfaz muestra ambos resultados lado a lado y permite inspeccionar probabilidades, señal `in_scope`, tokens y latencia.

Prueba al menos:

1. MFA no llega;
2. solicitud MFA que el usuario no inició;
3. VPN + Wi-Fi + USB;
4. reserva de sala.


## Conclusión

La pregunta correcta no es:

> ¿ML clásico o LLM?

La pregunta es:

> ¿Qué propiedades necesita esta decisión y cuál es la arquitectura más adecuada?

En algunos problemas, un modelo clásico pequeño es suficiente. En otros, la comprensión semántica y flexibilidad de un LLM aportan valor. Y en muchos sistemas empresariales, una arquitectura híbrida puede ser más razonable que elegir un único enfoque para todo.
